# Laboratorio 02 - Mineria de Datos

## Metodologia CRISP-DM aplicada

| Campo | Detalle |
|---|---|
| Estudiante | Jhadir Chahua Yupanqui |
| Curso | Mineria de Datos |
| Semana | 02 |
| Tema | Metodologia CRISP-DM aplicada |
| Dataset / caso | Caso real de mineria de datos |
| Docente | Pilar Rocio Sayan Mejia |
| Periodo | 2026-I |
---


---
# ACTIVIDAD 1: Revisión de Conceptos — Metodologías de Minería de Datos

Complete la siguiente tabla con definiciones propias basadas en lo estudiado en la clase teórica. **No copie textualmente** de los materiales; use sus propias palabras.

| N° | Concepto / Principio | Definición con sus propias palabras |
|:---:|---|---|
| 1 | CRISP-DM (definición y significado de las siglas) | |
| 2 | SEMMA (definición y significado de las siglas) | |
| 3 | Comprensión del negocio (Business Understanding) | |
| 4 | Comprensión de los datos (Data Understanding) | |
| 5 | Preparación de datos (Data Preparation) | |
| 6 | Modelado (Modeling) | |
| 7 | Evaluación (Evaluation) | |
| 8 | Despliegue (Deployment) | |
| 9 | ¿Cuál es la principal diferencia entre CRISP-DM y SEMMA? | |
| 10 | ¿Por qué CRISP-DM es un modelo cíclico e iterativo? | |

--
# ACTIVIDAD 2: Desarrollo Práctico — Aplicación de CRISP-DM en Google Colab

En esta actividad aplicarás las **6 fases de CRISP-DM** al dataset *Telco Customer Churn* que ya exploraste en el Laboratorio 1. El objetivo es documentar y ejecutar cada fase de forma estructurada.

> 📌 *Referencia general: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Caps. 2–3.*

## Paso 1: Fase 1 — Comprensión del Negocio (Business Understanding)

Antes de tocar los datos, debemos definir el problema de negocio con claridad.

> **Contexto:** TelcoPerú es una empresa de telecomunicaciones que enfrenta una tasa de abandono (churn) cercana al 26.5%. La gerencia quiere reducir el churn identificando clientes con mayor probabilidad de cancelar el servicio.

En esta fase se formulan las preguntas analíticas, los objetivos del proyecto y los criterios para evaluar si el modelo aporta valor al negocio.


### Pregunta 1
**¿Cuál es el problema de negocio? Formúlalo como una pregunta analítica clara.**

**Respuesta:** El problema de negocio es identificar qué clientes tienen mayor probabilidad de abandonar el servicio de TelcoPerú. Como pregunta analítica: **¿qué características del cliente y del servicio permiten predecir si un cliente hará churn para priorizar acciones de retención?**

### Pregunta 2
**¿Qué tipo de tarea de minería de datos es este problema: clasificación, regresión, clustering o asociación? Justifica tu respuesta.**

**Respuesta:** Es una tarea de **clasificación supervisada**, porque la variable objetivo `Churn` tiene dos categorías conocidas: `Yes` y `No`. El modelo aprende patrones históricos para asignar cada cliente a una de esas dos clases.

### Pregunta 3
**¿Cuáles serían los criterios de éxito para este proyecto? Menciona al menos 2 métricas.**

**Respuesta:** Un criterio de éxito técnico sería lograr un **recall alto para la clase Churn**, porque interesa detectar a la mayor cantidad posible de clientes que podrían abandonar. Otro criterio sería mantener un **accuracy o F1-score razonable**, evitando que el modelo genere demasiadas alertas incorrectas. Desde negocio, también se podría medir la reducción de la tasa de churn después de aplicar campañas de retención.


---
## 🔹 Paso 2: Fase 2 — Comprensión de los Datos (Data Understanding)

Carga el dataset y realiza un análisis exploratorio inicial para conocer los datos disponibles.

*Referencia: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Cap. 2.*

In [1]:
# Nota personal: se mantiene la base del laboratorio S02 y se documentan los pasos clave.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Cargar dataset
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Resumen general del dataset
print('=' * 60)
print('FASE 2: COMPRENSIÓN DE LOS DATOS')
print('=' * 60)
print(f'\nDimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nColumnas:\n{df.columns.tolist()}')

FASE 2: COMPRENSIÓN DE LOS DATOS

Dimensiones del dataset: 7043 filas × 21 columnas

Columnas:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
 #Tipos de datos
print('Tipos de datos:')
print(df.dtypes.value_counts())
print(f'\nVariables numéricas: {df.select_dtypes(include=[np.number]).columns.tolist()}')
print(f'\nVariables categóricas: {df.select_dtypes(include=["object"]).columns.tolist()}')

Tipos de datos:
object     18
int64       2
float64     1
Name: count, dtype: int64

Variables numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges']

Variables categóricas: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


In [3]:
# Valores nulos y problemas de calidad
print('Valores nulos por columna:')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('No se detectan valores nulos directamente.')
    print('\nPero revisemos TotalCharges (problema conocido del Lab 1):')
    print(f'Espacios en blanco en TotalCharges: {(df["TotalCharges"] == " ").sum()}')

Valores nulos por columna:
Series([], dtype: int64)
No se detectan valores nulos directamente.

Pero revisemos TotalCharges (problema conocido del Lab 1):
Espacios en blanco en TotalCharges: 11


In [ ]:
# Distribución de la variable objetivo
print('Distribución de Churn:')
print(df['Churn'].value_counts())
print('\nPorcentaje de churn:')
print(df['Churn'].value_counts(normalize=True) * 100)

# Gráfico con porcentajes para interpretar el desbalance de clases
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = churn_counts.plot(kind='bar', color=['#2E75B6', '#E65100'], ax=ax)
ax.set_title('Distribución de Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Churn')
ax.set_ylabel('Cantidad de clientes')

for idx, value in enumerate(churn_counts):
    ax.text(idx, value + 80, f'{value}\n({churn_pct.iloc[idx]:.1f}%)',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print('Comentario: la clase No Churn domina el dataset, por lo que el modelo puede tender a predecir mejor a los clientes que permanecen que a los clientes que abandonan.')


### Comentario adicional: variables numericas frente al churn

Para complementar el EDA, se comparan `tenure` y `MonthlyCharges` según la variable `Churn`. Esto ayuda a observar si los clientes que abandonan tienen patrones distintos de antigüedad o cargos mensuales.


In [ ]:
# Comparación de variables numéricas clave por estado de churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='Churn', y='tenure', hue='Churn', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Antigüedad del cliente vs Churn')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Meses como cliente')

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', hue='Churn', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Cargo mensual vs Churn')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Cargo mensual')

plt.tight_layout()
plt.show()

print('Comentario: los clientes con churn suelen mostrar menor antigüedad y cargos mensuales relativamente altos, señales útiles para priorizar alertas de retención.')


### Pregunta 4
**¿Cuántas variables tiene el dataset? Clasifícalas en numéricas y categóricas (indica cantidad de cada tipo).**

**Respuesta:** El dataset tiene **21 columnas** y **7043 registros**. Al cargarlo inicialmente, pandas identifica **3 variables numéricas** (`SeniorCitizen`, `tenure`, `MonthlyCharges`) y **18 variables categóricas o tipo texto**. Dentro de estas últimas está `TotalCharges`, que debería ser numérica, pero se carga como texto por contener espacios en blanco.

### Pregunta 5
**¿Identificas problemas de calidad en los datos? ¿Cuáles? (valores nulos, tipos incorrectos, inconsistencias).**

**Respuesta:** No aparecen valores nulos directamente con `isnull()`, pero sí se identifican **11 espacios en blanco en `TotalCharges`**. Ese es un problema de calidad porque impide tratar la variable como numérica. También se elimina `customerID` porque es un identificador y no representa un patrón generalizable para el modelo.


### Pregunta 6
**¿El dataset está desbalanceado? ¿Cómo podría afectar esto al entrenamiento del modelo?**

**Respuesta:** Sí, está desbalanceado. La clase `No` representa aproximadamente **73.46%** de los clientes y `Yes` aproximadamente **26.54%**. Esto puede hacer que el modelo aprenda mejor la clase mayoritaria y falle al detectar clientes que sí abandonan. Por eso, además del accuracy, conviene revisar métricas como **recall, precision y F1-score** para la clase `Churn = Yes`.


---
## 🔹 Paso 3: Fase 3 — Preparación de Datos (Data Preparation)

Ejecuta el siguiente pipeline de preparación de datos. Esta fase consume entre el **60% y 80%** del tiempo de un proyecto real.

*Referencia: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Cap. 2.*

In [ ]:
print('=' * 60)
print('FASE 3: PREPARACIÓN DE DATOS')
print('=' * 60)

# === LIMPIEZA ===
# Corregir TotalCharges: viene como texto porque algunos registros tienen espacios en blanco.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'\nValores nulos en TotalCharges después de conversión: {df["TotalCharges"].isnull().sum()}')

# Imputar nulos con la mediana sin usar inplace, para evitar problemas con pandas moderno.
mediana_total_charges = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(mediana_total_charges)
print(f'Mediana usada para imputar TotalCharges: {mediana_total_charges:.2f}')
print(f'Valores nulos después de imputación: {df["TotalCharges"].isnull().sum()}')
print('\nLimpieza completada')


In [6]:
# === TRANSFORMACIÓN ===
# Codificar variable objetivo
df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Variable objetivo codificada: Churn_num (Yes=1, No=0)')

# Eliminar columna customerID (no aporta al modelo)
df_model = df.drop(columns=['customerID', 'Churn'])
print(f'\nColumna customerID eliminada')
print(f'Dimensiones actuales: {df_model.shape}')

Variable objetivo codificada: Churn_num (Yes=1, No=0)

Columna customerID eliminada
Dimensiones actuales: (7043, 20)


In [7]:
# === CODIFICACIÓN ===
# Codificar variables categóricas con One-Hot Encoding
# separar variables numéricas y categóricas
num_cols = df_model.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df_model.select_dtypes(include=['object']).columns

# One-Hot Encoding solo para categóricas
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

print(f'Dimensiones después de One-Hot Encoding: {df_encoded.shape}')
print(f'\nPrimeras 10 columnas del dataset codificado:')
print(df_encoded.columns[:10].tolist())
print(f'\nÚltimas 10 columnas:')
print(df_encoded.columns[-10:].tolist())

print(f'\n✅ Preparación de datos completada')
print(f'Dataset listo para modelado: {df_encoded.shape[0]} registros × {df_encoded.shape[1]} variables')

Dimensiones después de One-Hot Encoding: (7043, 31)

Primeras 10 columnas del dataset codificado:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service']

Últimas 10 columnas:
['StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

✅ Preparación de datos completada
Dataset listo para modelado: 7043 registros × 31 variables


### Pregunta 6
**¿Por qué eliminamos la columna `customerID` antes del modelado? ¿Qué pasaría si la dejamos?**

**Respuesta:** Se elimina porque `customerID` solo identifica a cada cliente y no describe su comportamiento, contrato o consumo. Si se deja, el modelo podría memorizar identificadores del conjunto de entrenamiento, generando ruido y sobreajuste sin aportar valor para predecir clientes nuevos.

### Pregunta 7
**¿Qué es One-Hot Encoding y por qué es necesario para variables categóricas? ¿Cuántas columnas se generaron?**

**Respuesta:** One-Hot Encoding transforma categorías de texto en columnas binarias de 0 y 1. Es necesario porque los algoritmos de scikit-learn trabajan con variables numéricas. Después de la codificación, el dataset quedó con **31 columnas** en total: 30 variables predictoras y la variable objetivo `Churn_num`.


---
## 🔹 Paso 4: Fase 4 — Modelado (Modeling)

Aplica un modelo de clasificación para predecir el churn. Usaremos un **árbol de decisión** como primer modelo.

*Referencia: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Cap. 3.*

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('=' * 60)
print('FASE 4: MODELADO')
print('=' * 60)

# Separar features (X) y target (y)
X = df_encoded.drop(columns=['Churn_num'])
y = df_encoded['Churn_num']

print(f'\nFeatures (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'\nDistribución del target:')
print(y.value_counts())

FASE 4: MODELADO

Features (X): (7043, 30)
Target (y): (7043,)

Distribución del target:
Churn_num
0    5174
1    1869
Name: count, dtype: int64


In [9]:
# Dividir en entrenamiento (70%) y prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} registros ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Conjunto de prueba: {X_test.shape[0]} registros ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nDistribución en entrenamiento:')
print(y_train.value_counts(normalize=True) * 100)
print(f'\nDistribución en prueba:')
print(y_test.value_counts(normalize=True) * 100)

Conjunto de entrenamiento: 4930 registros (70%)
Conjunto de prueba: 2113 registros (30%)

Distribución en entrenamiento:
Churn_num
0    73.46856
1    26.53144
Name: proportion, dtype: float64

Distribución en prueba:
Churn_num
0    73.450071
1    26.549929
Name: proportion, dtype: float64


In [17]:
# Entrenar árbol de decisión
modelo_arbol = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo_arbol.fit(X_train, y_train)

# Predicciones
y_pred = modelo_arbol.predict(X_test)

print('✅ Modelo entrenado exitosamente')
print(f'\nParámetros del modelo:')
print(f'  Algoritmo: Árbol de Decisión')
print(f'  Profundidad máxima: {modelo_arbol.max_depth}')
print(f'  Nodos en el árbol: {modelo_arbol.tree_.node_count}')
print(f'  Features utilizados: {modelo_arbol.n_features_in_}')

✅ Modelo entrenado exitosamente

Parámetros del modelo:
  Algoritmo: Árbol de Decisión
  Profundidad máxima: 5
  Nodos en el árbol: 63
  Features utilizados: 30


### Pregunta 8
**¿Por qué dividimos los datos en entrenamiento y prueba? ¿Qué porcentaje usamos para cada conjunto?**

**Respuesta:** Dividimos los datos para entrenar el modelo con una parte y evaluar su desempeño con datos que no vio durante el aprendizaje. Esto permite estimar mejor cómo funcionaría con clientes nuevos. Se usó **70% para entrenamiento** y **30% para prueba**, manteniendo la proporción de churn mediante `stratify=y`.

### Pregunta 9
**¿Qué significa el parámetro `max_depth=5` en el árbol de decisión? ¿Qué pasaría si no lo limitamos?**

**Respuesta:** `max_depth=5` limita la profundidad máxima del árbol a cinco niveles. Esto controla la complejidad del modelo y reduce el riesgo de sobreajuste. Si no se limita, el árbol puede crecer demasiado, memorizar casos específicos del entrenamiento y perder capacidad de generalizar.


---
## 🔹 Paso 5: Fase 5 — Evaluación (Evaluation)

Evalúa el rendimiento del modelo con métricas de clasificación. No basta con que funcione técnicamente: debe cumplir los **objetivos del negocio** definidos en la Fase 1.

*Referencia: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Cap. 3.*

In [18]:
print('=' * 60)
print('FASE 5: EVALUACIÓN DEL MODELO')
print('=' * 60)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

FASE 5: EVALUACIÓN DEL MODELO

📊 Accuracy: 0.7880 (78.80%)


In [19]:
# Reporte de clasificación completo
print('Reporte de clasificación:')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['No Churn (0)', 'Churn (1)']))

Reporte de clasificación:
              precision    recall  f1-score   support

No Churn (0)       0.84      0.89      0.86      1552
   Churn (1)       0.62      0.52      0.56       561

    accuracy                           0.79      2113
   macro avg       0.73      0.70      0.71      2113
weighted avg       0.78      0.79      0.78      2113



In [ ]:
# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'], ax=ax,
            annot_kws={'size': 16})
ax.set_title('Matriz de Confusión — Árbol de Decisión', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Valor Real', fontsize=12)
plt.tight_layout()
plt.show()

# Interpretación numérica
print(f'Verdaderos Negativos (TN): {cm[0][0]} — Clientes que NO cancelaron, predichos correctamente')
print(f'Falsos Positivos (FP):     {cm[0][1]} — Clientes que NO cancelaron, pero el modelo dijo que sí')
print(f'Falsos Negativos (FN):     {cm[1][0]} — Clientes que SÍ cancelaron, pero el modelo NO los detectó')
print(f'Verdaderos Positivos (TP): {cm[1][1]} — Clientes que SÍ cancelaron, detectados correctamente')
print('\nComentario: para negocio, los falsos negativos son críticos porque representan clientes en riesgo que no recibirían acciones de retención.')


### Pregunta 10
**¿Cuál es el accuracy del modelo? ¿Es suficiente para considerar el modelo útil? Justifica.**

**Respuesta:** El accuracy obtenido es aproximadamente **78.80%**. Es un resultado inicial aceptable, pero no basta para concluir que el modelo es útil, porque el dataset está desbalanceado. Para churn importa especialmente detectar a quienes sí abandonan, y en esa clase el desempeño es menor que en `No Churn`.

### Pregunta 11
**Observa la matriz de confusión: ¿Cuántos falsos negativos hay? ¿Por qué son especialmente problemáticos en un caso de predicción de churn?**

**Respuesta:** Hay **271 falsos negativos**. Son problemáticos porque son clientes que realmente abandonaron, pero el modelo los clasificó como si fueran a permanecer. En la práctica, la empresa no los priorizaría para campañas de retención y perdería la oportunidad de intervenir a tiempo.

### Pregunta 12
**¿Qué métrica consideras más importante para este caso: precision o recall del churn? ¿Por qué?**

**Respuesta:** Considero más importante el **recall de la clase Churn**, porque mide qué proporción de clientes que abandonan logra detectar el modelo. En retención suele ser preferible revisar algunos falsos positivos antes que dejar sin atención a clientes con alto riesgo real de fuga.


---
## 🔹 Paso 6: Fase 6 — Despliegue (Deployment)

En un proyecto real, esta fase implica integrar el modelo en producción. Para este laboratorio, identificaremos las **variables más importantes** y haremos recomendaciones de negocio.

*Referencia: Provost, F. y Fawcett, T. (2013). Data Science for Business. O'Reilly Media. Cap. 2.*

In [ ]:
print('=' * 60)
print('FASE 6: DESPLIEGUE — VARIABLES MÁS IMPORTANTES')
print('=' * 60)

# Variables más importantes del modelo
importances = pd.Series(modelo_arbol.feature_importances_, index=X.columns)
top_features = importances.nlargest(10)

# Gráfico de importancia de variables
fig, ax = plt.subplots(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color='#2E75B6', ax=ax)
ax.set_title('Top 10 Variables Más Importantes para Predecir Churn',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia', fontsize=12)

# Marcar visualmente las tres variables con mayor peso.
for patch in ax.patches[-3:]:
    patch.set_color('#E65100')

plt.tight_layout()
plt.show()

print('Comentario: las tres variables principales concentran gran parte de la decisión del árbol; por eso conviene analizarlas antes de proponer acciones de retención.')


In [15]:
# Detalle de las variables más importantes
print('\nTop 10 predictores de churn:')
print('-' * 40)
for i, (feat, imp) in enumerate(top_features.items(), 1):
    print(f'  {i:2d}. {feat:<35s} {imp:.4f}')

# Resumen de perfiles de riesgo
print('\n' + '=' * 60)
print('PERFIL DE CLIENTE EN RIESGO DE CHURN')
print('=' * 60)
print('Basado en los resultados del modelo, un cliente con alto')
print('riesgo de abandono tiende a tener estas características:')
print('  → Contrato mes a mes (sin compromiso a largo plazo)')
print('  → Baja antigüedad (pocos meses como cliente)')
print('  → Cargos mensuales altos')
print('  → Sin servicios adicionales de protección')


Top 10 predictores de churn:
----------------------------------------
   1. tenure                              0.4027
   2. InternetService_Fiber optic         0.3222
   3. TotalCharges                        0.0632
   4. OnlineBackup_No internet service    0.0412
   5. PaymentMethod_Electronic check      0.0371
   6. Contract_Two year                   0.0327
   7. Contract_One year                   0.0310
   8. PaperlessBilling_Yes                0.0186
   9. MultipleLines_Yes                   0.0149
  10. MonthlyCharges                      0.0101

PERFIL DE CLIENTE EN RIESGO DE CHURN
Basado en los resultados del modelo, un cliente con alto
riesgo de abandono tiende a tener estas características:
  → Contrato mes a mes (sin compromiso a largo plazo)
  → Baja antigüedad (pocos meses como cliente)
  → Cargos mensuales altos
  → Sin servicios adicionales de protección


### Pregunta 13
**¿Cuáles son las 3 variables más importantes para predecir el churn según el modelo? ¿Tienen sentido desde el punto de vista del negocio?**

**Respuesta:** Las tres variables más importantes son **`tenure`**, **`InternetService_Fiber optic`** y **`TotalCharges`**. Sí tienen sentido desde negocio: la antigüedad refleja fidelización, el tipo de servicio de internet puede relacionarse con experiencia o costo del cliente, y el total pagado resume parte de la relación económica acumulada con la empresa.


---
# ACTIVIDAD 3: Caso de Estudio — Documento CRISP-DM

Basándote en **todo el trabajo realizado** en la Actividad 2, elabora un documento resumen del proyecto CRISP-DM.

> 📌 El documento debe seguir la estructura de las 6 fases y demostrar que comprendes cómo **cada fase se conecta con la siguiente**.

### Pregunta A
**Fase 1 — Comprensión del Negocio:** Describe el problema, los objetivos, los stakeholders y los criterios de éxito del proyecto.

**Respuesta:** El problema consiste en anticipar qué clientes de TelcoPerú tienen mayor probabilidad de abandonar el servicio. El objetivo es apoyar decisiones de retención usando datos históricos del cliente, contrato, facturación y servicios. Los stakeholders serían gerencia comercial, marketing, atención al cliente y el equipo de analítica. Los criterios de éxito incluyen aumentar el recall de clientes con churn, lograr un F1-score aceptable para esa clase y reducir la tasa de abandono mediante campañas dirigidas.

### Pregunta B
**Fase 2 — Comprensión de los Datos:** Describe el dataset, las variables más relevantes, los problemas de calidad encontrados y los hallazgos del EDA.

**Respuesta:** El dataset contiene 7043 registros y 21 columnas. Las variables relevantes incluyen `tenure`, `Contract`, `MonthlyCharges`, `TotalCharges`, `InternetService`, `PaymentMethod` y `Churn`. El problema de calidad más claro fue `TotalCharges`, que se cargó como texto porque tenía 11 espacios en blanco. El EDA mostró que el dataset está desbalanceado: la mayoría de clientes no presenta churn, mientras que cerca de una cuarta parte sí abandona.

### Pregunta C
**Fase 3 — Preparación de Datos:** Explica qué transformaciones realizaste y por qué. Menciona limpieza, codificación y selección de variables.

**Respuesta:** Se convirtió `TotalCharges` a numérico y se imputaron los valores faltantes con la mediana. Luego se creó `Churn_num` como variable objetivo binaria, se retiró `customerID` porque solo identifica clientes y se aplicó One-Hot Encoding a las variables categóricas. Estas transformaciones dejan el dataset en formato numérico para que pueda ser usado por scikit-learn.

### Pregunta D
**Fase 4 — Modelado:** ¿Qué algoritmo usaste? ¿Qué hiperparámetros configuraste? ¿Por qué elegiste ese modelo?

**Respuesta:** Se utilizó un árbol de decisión (`DecisionTreeClassifier`) con `max_depth=5` y `random_state=42`. Elegí este modelo porque funciona bien como línea base, es interpretable y permite revisar qué variables tienen mayor peso en la predicción. La profundidad máxima ayuda a controlar el sobreajuste.

### Pregunta E
**Fase 5 — Evaluación:** ¿Qué métricas obtuviste? ¿El modelo cumple los objetivos de negocio definidos en la Fase 1? ¿Qué limitaciones tiene?

**Respuesta:** El modelo obtuvo un accuracy aproximado de 78.80%. Para la clase churn, el recall fue cercano a 0.52 y el F1-score a 0.56. El modelo sirve como primera aproximación, pero todavía tiene limitaciones para negocio porque deja 271 falsos negativos. Esto significa que varios clientes que sí abandonan no serían detectados a tiempo.

### Pregunta F
**Fase 6 — Despliegue:** ¿Cómo implementarías este modelo en la empresa? Propón un sistema de alertas tempranas y 3 acciones concretas de retención basadas en los hallazgos.

**Respuesta:** Implementaría el modelo como un scoring semanal de clientes activos. Los clientes con mayor probabilidad de churn pasarían a una lista de alertas para campañas comerciales. Como acciones de retención propondría: ofrecer beneficios a clientes nuevos o con bajo `tenure`, revisar descuentos o paquetes para clientes con cargos mensuales altos y crear campañas específicas para clientes con servicios o contratos asociados a mayor riesgo.


---
# CONCLUSIONES

1. La metodología CRISP-DM ayuda a ordenar el proyecto desde el problema de negocio hasta la propuesta de despliegue, evitando empezar directamente por el modelo sin entender los datos.

2. La preparación de datos fue clave: `TotalCharges` parecía una variable categórica por problemas de formato, pero al corregirla se pudo usar adecuadamente en el modelo.

3. El árbol de decisión logró un accuracy cercano al 79%, pero el recall de churn muestra que todavía se deben mejorar los falsos negativos antes de usarlo como herramienta final de retención.


---
### 📚 Referencias bibliográficas

- Berry, M. J. A. y Linoff, G. S. (2004). *Data Mining Techniques: For Marketing, Sales, and Customer Relationship Management* (2.ª ed.). Wiley.
- Hernández, J., Ramírez, M. J. y Ferri, C. (2004). *Introducción a la Minería de Datos*. Pearson Educación.
- Provost, F. y Fawcett, T. (2013). *Data Science for Business*. O'Reilly Media.
- Gutman, A. J. y Goldmeier, J. (2021). *Becoming a Data Head*. Wiley.

---
*Minería de Datos — Semana 2 | TECSUP 2026-I | Prof. Pilar Rocío Sayán Mejía*